# TP4 — RAG (Retrieval-Augmented Generation) appliqué à mon CV

**Étudiant :** Aguelloul Mouad — Master IA & Digital Computing, FST Béni Mellal
**Module :** Système Distribué Agentique

Ce notebook reprend les étapes du notebook `rag.ipynb` fourni, appliquées à **mon CV** :

1. chargement du PDF et découpage en chunks ;
2. vectorisation (embeddings) et stockage dans ChromaDB ;
3. recherche des chunks pertinents (retriever) ;
4. construction du prompt et génération de la réponse ;
5. évaluation de la fidélité au contexte par un LLM juge ;
6. expérience : influence du nombre de chunks récupérés `k`.

**Modèles :** n'ayant pas de clé OpenAI, j'utilise l'API gratuite **Google Gemini** (autorisée dans la consigne) :

| | Notebook fourni | Ce notebook |
| :--- | :--- | :--- |
| LLM du RAG | `gpt-4o-mini` | `gemini-3.5-flash-lite` |
| LLM juge | `gpt-4o` | `gemini-3.5-flash` |
| Embeddings | `text-embedding-3-small` | `gemini-embedding-001` |
| Vector store | `langchain_community` Chroma | `langchain_chroma` Chroma |

# 0. Environnement

```
uv init
uv venv
.venv\Scripts\activate
uv add ipykernel langchain langchain-text-splitters langchain-community python-dotenv pypdf tiktoken
uv add langchain-google-genai chromadb langchain-chroma
```
Clé API dans un fichier `.env` (non publié, voir `.gitignore`) : `GOOGLE_API_KEY=...`

In [1]:
import os
import shutil
import tiktoken
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

load_dotenv(override=True)
assert os.getenv("GOOGLE_API_KEY"), "GOOGLE_API_KEY manquante dans .env"

C:\Users\Administrator\AppData\Local\Temp\ipykernel_25056\2140931351.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## LLM

In [2]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0)
llm.invoke("Dis bonjour en un mot.").text

C:\Users\Administrator\Desktop\TP4\RAG_TP4_DSD\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'Bonjour.'

> Les modèles `gemini-3.5` ignorent `temperature` (valeur fixe) et renvoient leur réponse sous forme de blocs :
> `.text` donne directement le texte, là où le notebook fourni utilisait `.content`.

# 1. Chargement du PDF et chunking

In [3]:
pdf_file = "./pdfs/CV.pdf"
pdf_loader = PyPDFLoader(pdf_file)

pages = pdf_loader.load()
enc = tiktoken.get_encoding("o200k_base")
print("Pages :", len(pages))
print("Tokens :", sum(len(enc.encode(p.page_content)) for p in pages))

Pages : 1
Tokens : 716


Mon CV fait environ **700 tokens**. Avec les paramètres du notebook fourni (chunks de 300 tokens), il ne donnerait que
**3 chunks** : le retriever renverrait quasiment tout le document. Je prends des chunks de **100 tokens** (overlap 20).

In [4]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="o200k_base", chunk_size=100, chunk_overlap=20
)
chunks = pdf_loader.load_and_split(text_splitter)
for i, c in enumerate(chunks):
    c.metadata["chunk"] = i   # numéro du chunk, utile pour suivre ce que renvoie le retriever
print("Nombre de chunks :", len(chunks), "\n")
for i, c in enumerate(chunks):
    print(f"[{i}]", c.page_content.replace("\n", " | ")[:160], "...")

Nombre de chunks : 9 

[0] Aguelloul Mouad | Adresse: Safi, Maroc | Date de naissance: 03/06/2002 | Email: aguelloumouad@gmail.com | GitHub: github/aguelloulmouad-hub | Mobile: +212 708 0 ...
[1] Étudiant en Master IA & Digital Computing | DOMAINE DE COMPÉTENCE | •Intégration de solutions IA, machine learning et computer vision pour des cas d’usage métie ...
[2] •Analyse, visualisation et prétraitement de données pour l’aide à la décision. | PROJETS | FlowGuard – Détection de fraude en temps réelProjet personnel / acadé ...
[3] •Dashboard de supervision développé avec Dash/Plotly. | Détection des feux de forêt et de la fuméeProjet IA / vision par ordinateur - 2026 | •Modèle basé sur In ...
[4] Gestion médicale distribuéeProjet systèmes distribués - 2026 | •Système NoSQL basé sur Apache Cassandra. | •Conception Query-First et backend Python modulaire.  ...
[5] Gestion de soutenance doctorale sous OdooProjet ERP - 2024 | •Automatisation du workflow de soutenance. | •Notifications et géné

**Observation sur l'extraction du texte :** `pypdf` colle certains mots dont la séparation n'était que visuelle dans le
PDF (mise en page en colonnes) : *« temps réel**Projet** personnel »*, *« **Programmation**Python »*, ou coupe des mots
(*« Infor- matique »*). Le sens reste compréhensible pour le modèle d'embedding, mais c'est une source de bruit à connaître
pour un RAG sur des PDF mis en page.

# 2. Embeddings et ChromaDB

In [5]:
embedding_model = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
print("Dimension d'un embedding :", len(embedding_model.embed_query("test")))

shutil.rmtree("./store", ignore_errors=True)   # repartir d'une base vide à chaque exécution
vectorstore = Chroma.from_documents(
    chunks, embedding_model, collection_name="cv_mouad", persist_directory="./store"
)
print("Vecteurs stockés :", vectorstore._collection.count())

Dimension d'un embedding : 3072


Vecteurs stockés : 9


# 3. Retriever

Le notebook fourni écrit `search_Kwargs={'k',5}` : le nom du paramètre est mal orthographié (majuscule) et `{'k',5}` est un
*set*, pas un dictionnaire. Le paramètre est donc ignoré et LangChain garde sa valeur par défaut `k=4` (d'où les 4 chunks
affichés dans le notebook fourni). Écriture correcte : `search_kwargs={"k": 3}`.

In [6]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

docs = retriever.invoke("Quelle accuracy le modèle de détection des feux de forêt a-t-il obtenue ?")
print("Chunks récupérés :", [d.metadata["chunk"] for d in docs])
for d in docs:
    print("-", d.page_content.replace("\n", " | ")[:200])

Chunks récupérés : [3, 7, 2]
- •Dashboard de supervision développé avec Dash/Plotly. | Détection des feux de forêt et de la fuméeProjet IA / vision par ordinateur - 2026 | •Modèle basé sur Inception-ResNet-v2 avec transfert d’appre
- IA / Machine Learning Machine Learning, Deep Learning, CNN, Transfer Learning, Federated Learning, PyTorch, TensorFlow, | RAG, Fine-tuning, NLP/OCR, scikit-learn | Computer VisionOpenCV, YOLO, classif
- •Analyse, visualisation et prétraitement de données pour l’aide à la décision. | PROJETS | FlowGuard – Détection de fraude en temps réelProjet personnel / académique - 2026 | •Pipeline Kafka pour l’in


# 4. Prompt

In [7]:
prompt_template = """
Answer the following question based only on provided context.
The context is an extract of the CV of Aguelloul Mouad, a Master student in AI & Digital Computing (Morocco).
Answer in French.
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!
<context>
{context}
</context>
<question>
{question}
</question>
"""

In [8]:
question = "Quelle accuracy le modèle de détection des feux de forêt a-t-il obtenue ?"
context = ". ".join(d.page_content for d in retriever.invoke(question))
prompt = prompt_template.format(context=context, question=question)
print(prompt)


Answer the following question based only on provided context.
The context is an extract of the CV of Aguelloul Mouad, a Master student in AI & Digital Computing (Morocco).
Answer in French.
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!
<context>
•Dashboard de supervision développé avec Dash/Plotly.
Détection des feux de forêt et de la fuméeProjet IA / vision par ordinateur - 2026
•Modèle basé sur Inception-ResNet-v2 avec transfert d’apprentissage.
•Prétraitement et détection d’images/vidéos avec OpenCV.
•Performances élevées avec une accuracy de 99,10%.
Gestion médicale distribuéeProjet systèmes distribués - 2026. IA / Machine Learning Machine Learning, Deep Learning, CNN, Transfer Learning, Federated Learning, PyTorch, TensorFlow,
RAG, Fine-tuning, NLP/OCR, scikit-learn
Computer VisionOpenCV, YOLO, classification d’images, détection d’objets, traitement vidéo temps réel
Big

# 5. Réponse du LLM

In [9]:
display(Markdown(llm.invoke(prompt).text))

C:\Users\Administrator\Desktop\TP4\RAG_TP4_DSD\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Le modèle de détection des feux de forêt a obtenu une accuracy de 99,10%.

# 6. Fonction RAG

In [10]:
def RAG(query, llm=llm, prompt_template=prompt_template, retriever=retriever):
    context_docs = retriever.invoke(query)
    context = ". ".join(d.page_content for d in context_docs)
    prompt = prompt_template.format(context=context, question=query)
    return llm.invoke(prompt).text

Je pose 4 questions dont la réponse est dans mon CV et 2 questions dont la réponse n'y est pas.
Les réponses sont gardées dans `reponses` pour être réutilisées par le juge (§7) sans rappeler le LLM.

In [11]:
questions_cv = [
    "Quelles technologies sont utilisées dans le projet FlowGuard ?",
    "Avec quel framework le système multi-agents bataille navale a-t-il été développé ?",
    "Quelles bases de données maîtrise-t-il ?",
    "Quelle accuracy le modèle de détection des feux de forêt a-t-il obtenue ?",
]
questions_hors_cv = [
    "Quel est son niveau d'anglais ?",              # aucune section langues dans le CV
    "Qui a gagné la Coupe du monde de football 2022 ?",
]
reponses = {}
for q in questions_cv + questions_hors_cv:
    reponses[q] = RAG(q)
    display(Markdown(f"**{q}**\n\n{reponses[q]}\n\n---"))

C:\Users\Administrator\Desktop\TP4\RAG_TP4_DSD\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


**Quelles technologies sont utilisées dans le projet FlowGuard ?**

Basé sur le contexte fourni, les technologies utilisées dans le projet FlowGuard sont :

* **Kafka** (pour le pipeline d'ingestion et de traitement de flux)
* **ML** / Apprentissage fédéré **FedAvg** (pour le scoring)
* **Neo4j** (pour le stockage)
* **Dash/Plotly** (pour le dashboard de supervision)

---

C:\Users\Administrator\Desktop\TP4\RAG_TP4_DSD\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


**Avec quel framework le système multi-agents bataille navale a-t-il été développé ?**

Le système multi-agents bataille navale a été développé avec le framework JADE.

---

C:\Users\Administrator\Desktop\TP4\RAG_TP4_DSD\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


...

**Quelle accuracy le modèle de détection des feux de forêt a-t-il obtenue ?**

Le modèle de détection des feux de forêt a obtenu une accuracy de 99,10%.

---

C:\Users\Administrator\Desktop\TP4\RAG_TP4_DSD\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


**Quel est son niveau d'anglais ?**

Je ne sais pas!

---

C:\Users\Administrator\Desktop\TP4\RAG_TP4_DSD\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


**Qui a gagné la Coupe du monde de football 2022 ?**

Je ne sais pas!

---

# 7. Évaluation par un LLM juge

![Architecture](./rag_juge.gif)

In [12]:
user_message_template = """
###Question
{question}
###Context
{context}
###Answer
{answer}
"""

groundness_rater_system_message = """
Vous êtes chargé d'évaluer des réponses générées par une IA à des questions posées par des utilisateurs.
On vous présentera une question, le contexte utilisé par le système d'IA pour générer la réponse, ainsi qu'une réponse générée par l'IA à la question.
Dans l'entrée, la question commencera par ###Question, le contexte commencera par ###Context, et la réponse générée par L'IA commencera par ###Answer.
Critères d'évaluation :
La tâche consiste à juger dans quelle mesure la réponse respecte la métrique.

1- La métrique n'est pas respectée du tout
2- La métrique n'est respectée que dans une mesure limitée
3- La métrique est respectée dans une bonne mesure
4- La métrique est respectée en grande partie
5- La métrique est entièrement respectée

Métrique :

La réponse doit être dérivée uniquement des informations présentées dans le contexte.

Instructions:

Écrivez d'abord les étapes nécessaires pour évaluer la réponse selon la métrique.
Donnez une explication étape par étape indiquant si la réponse respecte la métrique, en considérant la question et le contexte comme entrées.
Évaluez ensuite dans quelle mesure la métrique est respectée.
Utilisez les informations précédentes pour noter la réponse selon les critères d'évaluation et attribuer un score.
Dernière ligne obligatoire : NOTE = x/5
"""

Le juge (`gemini-3.5-flash`) est plus puissant que le LLM du RAG. J'ai ajouté une dernière ligne obligatoire
`NOTE = x/5` au prompt du juge pour pouvoir extraire la note automatiquement.

In [13]:
juge = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)

def evaluate(system_message, user_message_template, question, answer, retriever=retriever, model=juge):
    context = ". ".join(d.page_content for d in retriever.invoke(question))
    prompt = f"""
     {system_message}\n
     USER:
     {user_message_template.format(question=question, context=context, answer=answer)}
     """
    return model.invoke(prompt).text

## 7.1 Évaluation détaillée d'une réponse

In [14]:
q = questions_cv[0]
display(Markdown(evaluate(groundness_rater_system_message, user_message_template, q, reponses[q])))

Voici l'évaluation de la réponse générée par l'IA.

### Étapes d'évaluation

1. **Identifier les faits et technologies énoncés dans la réponse** : Lister chaque technologie mentionnée par l'IA et son utilisation associative.
2. **Vérifier dans le contexte** : Comparer chaque point de la réponse spécifiquement avec la section du contexte décrivant le projet "FlowGuard".
3. **Détecter les informations non fondées (hallucinations)** : S'assurer qu'aucune technologie ou précision n'a été inventée ou tirée de connaissances externes.
4. **Conclure sur le respect de la métrique** : Déterminer si la réponse dérive exclusivement du contexte fourni.

---

### Explication étape par étape

* **Kafka** : Le contexte indique « *Pipeline Kafka pour l’ingestion et le traitement de flux.* ». La réponse inclut Kafka avec cette fonction exacte.
* **ML / Apprentissage fédéré FedAvg** : Le contexte indique « *Scoring ML, apprentissage fédéré FedAvg...* ». La réponse inclut ces éléments.
* **Neo4j** : Le contexte indique « *...et stockage Neo4j.* ». La réponse inclut Neo4j pour le stockage.
* **Dash/Plotly** : Le contexte indique « *Dashboard de supervision développé avec Dash/Plotly..* ». La réponse inclut Dash/Plotly pour le dashboard de supervision.

Toutes les informations présentes dans la réponse proviennent directement et uniquement des puces décrivant le projet FlowGuard dans le contexte. Aucune information externe n'a été ajoutée.

---

### Évaluation

La métrique exige que la réponse soit dérivée **uniquement** des informations présentées dans le contexte. La réponse s'appuie à 100 % sur les données du texte sans rien extrapoler ni ajouter. La métrique est donc entièrement respectée.

NOTE = 5/5

## 7.2 Tableau des notes

In [15]:
import re

def note(texte):
    m = re.search(r"NOTE\s*=\s*(\d)", texte)
    return f"{m.group(1)}/5" if m else "?"

lignes = ["| Question | Réponse du RAG | Note |", "|:--|:--|:--:|"]
for q in questions_cv[1:] + questions_hors_cv[:1]:
    jugement = evaluate(groundness_rater_system_message, user_message_template, q, reponses[q])
    lignes.append(f"| {q} | {reponses[q].strip().replace(chr(10), ' ')} | {note(jugement)} |")
display(Markdown("\n".join(lignes)))

| Question | Réponse du RAG | Note |
|:--|:--|:--:|
| Avec quel framework le système multi-agents bataille navale a-t-il été développé ? | Le système multi-agents bataille navale a été développé avec le framework JADE. | 5/5 |
| Quelles bases de données maîtrise-t-il ? | D'après le contexte fourni, les bases de données qu'il maîtrise sont : * MySQL * Oracle Database * Neo4j * Apache Cassandra | 5/5 |
| Quelle accuracy le modèle de détection des feux de forêt a-t-il obtenue ? | Le modèle de détection des feux de forêt a obtenu une accuracy de 99,10%. | 5/5 |
| Quel est son niveau d'anglais ? | Je ne sais pas! | 5/5 |

# 8. Expérience : influence de `k` sur une question de synthèse

Les questions précédentes portent sur **un seul** passage du CV. Une question de **synthèse** comme
*« Combien de projets sont présentés dans le CV et lesquels ? »* demande au contraire d'assembler des informations
réparties sur **plusieurs** chunks (mes 5 projets s'étendent des chunks 2 à 5). Je compare `k=1` et `k=3`.

In [16]:
q_synthese = "Combien de projets sont présentés dans le CV et lesquels ?"
resultats_k = {}
for k in (1, 3):
    r_k = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": k})
    docs_k = r_k.invoke(q_synthese)
    resultats_k[k] = (r_k, RAG(q_synthese, retriever=r_k))
    display(Markdown(f"### k = {k} — chunks récupérés : {[d.metadata['chunk'] for d in docs_k]}\n\n{resultats_k[k][1]}"))

C:\Users\Administrator\Desktop\TP4\RAG_TP4_DSD\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


### k = 1 — chunks récupérés : [4]

D'après le contexte fourni, il y a 3 projets présentés :

1. **Gestion médicale distribuée** (Projet systèmes distribués - 2026)
2. **Système multi-agents bataille navale** (Projet multi-agents - 2025)
3. **Gestion de soutenance doctorale sous Odoo** (Projet ERP - 2024)

C:\Users\Administrator\Desktop\TP4\RAG_TP4_DSD\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


### k = 3 — chunks récupérés : [4, 3, 2]

Basé sur le contexte fourni, il y a 6 projets présentés (en comptant les différentes mentions) :

1. **Gestion médicale distribuée** (Projet systèmes distribués - 2026) : Système NoSQL basé sur Apache Cassandra et conception Query-First avec backend Python modulaire. (Aussi mentionné pour l'analyse, la visualisation et le prétraitement de données pour l'aide à la décision).
2. **Système multi-agents bataille navale** (Projet multi-agents - 2025) : Développement avec JADE et communication ACL, interface graphique JavaFX et agents autonomes.
3. **Gestion de soutenance doctorale sous Odoo** (Projet ERP - 2024) : Dashboard de supervision développé avec Dash/Plotly.
4. **Détection des feux de forêt et de la fumée** (Projet IA / vision par ordinateur - 2026) : Modèle basé sur Inception-ResNet-v2, prétraitement/détection avec OpenCV et accuracy de 99,10%.
5. **Gestion médicale distribuée** (Projet systèmes distribués - 2026) : Analyse, visualisation et prétraitement de données pour l’aide à la décision.
6. **FlowGuard – Détection de fraude en temps réel** (Projet personnel / académique - 2026) : Pipeline Kafka, scoring ML, apprentissage fédéré FedAvg, stockage Neo4j et dashboard Dash/Plotly.

In [17]:
for k, (r_k, rep) in resultats_k.items():
    jugement = evaluate(groundness_rater_system_message, user_message_template, q_synthese, rep, retriever=r_k)
    display(Markdown(f"**k = {k} → note du juge : {note(jugement)}**"))

**k = 1 → note du juge : 5/5**

**k = 3 → note du juge : 5/5**

### Analyse
**k = 1** — le retriever ne renvoie que le chunk 4 : le RAG cite **3 projets** sur 5. La réponse est fidèle au contexte
mais **incomplète** : FlowGuard et la détection des feux de forêt se trouvent dans d'autres chunks.

**k = 3** — les chunks 4, 3 et 2 contiennent bien les 5 projets, mais le RAG répond **« 6 projets »** : il compte
*Gestion médicale distribuée* deux fois et rattache certaines puces au mauvais projet (le dashboard Dash/Plotly de
FlowGuard est attribué au projet Odoo). Deux causes :
1. **le chevauchement (overlap)** : la ligne *« Gestion médicale distribuée – Projet systèmes distribués »* est à la fois à
   la fin du chunk 3 et au début du chunk 4, elle apparaît donc deux fois dans le contexte ;
2. **l'ordre des chunks** : ils sont concaténés par ordre de similarité (4, 3, 2) et non dans l'ordre du document ; une puce
   qui termine un chunk se retrouve collée au titre de projet qui commence le chunk suivant dans le contexte.

**Le juge donne 5/5 dans les deux cas.** La métrique de *groundedness* vérifie seulement que la réponse vient du contexte :
elle ne mesure ni la **complétude** (k = 1) ni l'**exactitude** d'un comptage ou d'une attribution (k = 3).

**Pistes d'amélioration :** trier les chunks récupérés selon leur position dans le document et supprimer les doublons dus
à l'overlap avant de construire le contexte ; découper le CV par section (un chunk par projet) plutôt que par nombre de
tokens ; compléter l'évaluation par d'autres métriques (pertinence, complétude, exactitude par rapport à une réponse de
référence).